In [1]:
import sys
import subprocess

subprocess.check_call([
    sys.executable, "-m", "pip", "install", "--quiet", "--force-reinstall", "--no-cache-dir",
    "torch==2.7.1",
    "torchvision==0.22.1",
    "torchaudio==2.7.1",
    "transformers==4.52.4",
    "accelerate==1.7.0",
    "datasets==3.6.0",
    "evaluate==0.4.3",
    "tokenizers==0.21.1"
])


0

In [2]:
import sys
import transformers
import accelerate
import datasets
import torch

print("Python executable:", sys.executable)
print("Transformers:", transformers.__version__, transformers.__file__)
print("Accelerate:", accelerate.__version__, accelerate.__file__)
print("Datasets:", datasets.__version__)
print("Torch:", torch.__version__)

Python executable: /usr/bin/python3
Transformers: 4.52.4 /usr/local/lib/python3.12/dist-packages/transformers/__init__.py
Accelerate: 1.7.0 /usr/local/lib/python3.12/dist-packages/accelerate/__init__.py
Datasets: 3.6.0
Torch: 2.7.1+cu126


In [3]:
# =========================================================
# 1. IMPORTS
# =========================================================
import os
import csv
import time
import random
import numpy as np
import pandas as pd
import torch
import torch.nn as nn

from datasets import load_dataset, Dataset
from sklearn.metrics import f1_score, classification_report
from scipy.stats import pearsonr

from transformers import (
    RobertaTokenizer,
    RobertaForSequenceClassification,
    RobertaModel,
    TrainingArguments,
    Trainer,
    DataCollatorWithPadding,
    TrainerCallback,
    EarlyStoppingCallback
)


# =========================================================
# 2. GOOGLE DRIVE AND LOG PATHS
# =========================================================
from google.colab import drive
drive.mount("/content/drive")

BASE_LOG_DIR = "/content/drive/MyDrive/RoBERTa_Hierarchical_Logs"

LOG_FILE_STEP1 = os.path.join(
    BASE_LOG_DIR,
    "RoBERTa_Hierarchical_Step1.csv"
)

LOG_FILE_STEP2 = os.path.join(
    BASE_LOG_DIR,
    "RoBERTa_Hierarchical_Step2.csv"
)

STEP1_SENTENCE_DIR = os.path.join(
    BASE_LOG_DIR,
    "Step1_Sentence_Logs"
)

STEP2_SENTENCE_DIR = os.path.join(
    BASE_LOG_DIR,
    "Step2_Combined_Sentence_Logs"
)

for directory in [
    BASE_LOG_DIR,
    STEP1_SENTENCE_DIR,
    STEP2_SENTENCE_DIR
]:
    os.makedirs(directory, exist_ok=True)

with open(LOG_FILE_STEP1, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "roberta-base"])
    writer.writerow(["step", "STEP 1"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 16])
    writer.writerow(["eval_batch_size", 16])
    writer.writerow(["epochs", 2])
    writer.writerow([])

with open(LOG_FILE_STEP2, "w", newline="", encoding="utf-8") as file:
    writer = csv.writer(file)
    writer.writerow(["model", "roberta-base"])
    writer.writerow(["step", "STEP 2"])
    writer.writerow(["learning_rate", 2e-5])
    writer.writerow(["train_batch_size", 8])
    writer.writerow(["eval_batch_size", 8])
    writer.writerow(["epochs", 10])
    writer.writerow([])

print("Step 1 results file:", LOG_FILE_STEP1)
print("Step 2 results file:", LOG_FILE_STEP2)
print("Step 1 sentence directory:", STEP1_SENTENCE_DIR)
print("Step 2 sentence directory:", STEP2_SENTENCE_DIR)


# =========================================================
# 3. SEED AND DEVICE
# =========================================================
SEED = 42

def set_seed(seed=42):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)

    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

set_seed(SEED)

device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

print("Using device:", device)


# =========================================================
# 4. LOAD DATASET
# =========================================================
print("\nLoading BRIGHTER dataset...")

train_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="train"
)

val_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="dev"
)

test_data = load_dataset(
    "brighter-dataset/BRIGHTER-emotion-intensities",
    "eng",
    split="test"
)

train_df = train_data.to_pandas()
val_df = val_data.to_pandas()
test_df = test_data.to_pandas()

print("Original sizes:")
print({
    "train": len(train_df),
    "val": len(val_df),
    "test": len(test_df)
})


# =========================================================
# 5. LABEL CONFIGURATION
# =========================================================
EMOTIONS = [
    "anger",
    "fear",
    "joy",
    "sadness",
    "surprise"
]

INTENSITY_COLUMNS = [
    f"{emotion}_intensity"
    for emotion in EMOTIONS
]

LEVELS = [1, 2, 3]

LABELS = [
    f"{emotion}_{level}"
    for emotion in EMOTIONS
    for level in LEVELS
]

print("\nEmotion labels:")
print(EMOTIONS)

print("\nCombined emotion-intensity labels:")
print(LABELS)


# =========================================================
# 6. PREPARE TWO-STEP DATA
# =========================================================
def prepare_two_step_data(df):
    df = df.copy()

    if "disgust" in df.columns:
        df = df.drop(columns=["disgust"])

    for emotion in EMOTIONS:
        df[f"{emotion}_intensity"] = df[emotion].astype(int)

    for emotion in EMOTIONS:
        df[emotion] = (
            df[f"{emotion}_intensity"] > 0
        ).astype(int)

    return df


train_two = prepare_two_step_data(train_df)
val_two = prepare_two_step_data(val_df)
test_two = prepare_two_step_data(test_df)


# =========================================================
# 7. COMBINE AND REDISTRIBUTE: 70 / 20 / 10
# =========================================================
full_df = pd.concat(
    [train_two, val_two, test_two],
    ignore_index=True
)

full_df = full_df[
    ["text"] + EMOTIONS + INTENSITY_COLUMNS
]

full_df = full_df.sample(
    frac=1,
    random_state=SEED
).reset_index(drop=True)

number_of_rows = len(full_df)

train_end = int(0.7 * number_of_rows)
val_end = int(0.9 * number_of_rows)

train_split = full_df[
    :train_end
].reset_index(drop=True)

val_split = full_df[
    train_end:val_end
].reset_index(drop=True)

test_split = full_df[
    val_end:
].reset_index(drop=True)

print("\nRedistributed split sizes:")
print({
    "train": len(train_split),
    "val": len(val_split),
    "test": len(test_split)
})

print("\nSample data:")
print(train_split.head())

test_texts = test_split["text"].tolist()


# =========================================================
# 8. STEP 1 DATA: EMOTION PRESENCE
# =========================================================
train_step1_df = train_split[
    ["text"] + EMOTIONS
].copy()

val_step1_df = val_split[
    ["text"] + EMOTIONS
].copy()

test_step1_df = test_split[
    ["text"] + EMOTIONS
].copy()


# =========================================================
# 9. STEP 2 DATA: INTENSITY PREDICTION
# =========================================================
train_step2_df = train_split[
    ["text"] + INTENSITY_COLUMNS
].copy()

val_step2_df = val_split[
    ["text"] + INTENSITY_COLUMNS
].copy()

test_step2_df = test_split[
    ["text"] + INTENSITY_COLUMNS
].copy()


# =========================================================
# 10. CONVERT TO HUGGING FACE DATASETS
# =========================================================
train_step1_ds = Dataset.from_pandas(
    train_step1_df,
    preserve_index=False
)

val_step1_ds = Dataset.from_pandas(
    val_step1_df,
    preserve_index=False
)

test_step1_ds = Dataset.from_pandas(
    test_step1_df,
    preserve_index=False
)

train_step2_ds = Dataset.from_pandas(
    train_step2_df,
    preserve_index=False
)

val_step2_ds = Dataset.from_pandas(
    val_step2_df,
    preserve_index=False
)

test_step2_ds = Dataset.from_pandas(
    test_step2_df,
    preserve_index=False
)


# =========================================================
# 11. TOKENIZATION
# =========================================================
tokenizer = RobertaTokenizer.from_pretrained(
    "roberta-base"
)

data_collator = DataCollatorWithPadding(
    tokenizer=tokenizer
)

def tokenize_function(examples):
    return tokenizer(
        examples["text"],
        truncation=True,
        max_length=128
    )


train_step1_ds = train_step1_ds.map(
    tokenize_function,
    batched=True
)

val_step1_ds = val_step1_ds.map(
    tokenize_function,
    batched=True
)

test_step1_ds = test_step1_ds.map(
    tokenize_function,
    batched=True
)

train_step2_ds = train_step2_ds.map(
    tokenize_function,
    batched=True
)

val_step2_ds = val_step2_ds.map(
    tokenize_function,
    batched=True
)

test_step2_ds = test_step2_ds.map(
    tokenize_function,
    batched=True
)


# =========================================================
# 12. ADD LABEL VECTORS
# =========================================================
def add_step1_labels(example):
    example["labels"] = [
        float(example[emotion])
        for emotion in EMOTIONS
    ]

    return example


def add_step2_labels(example):
    example["labels"] = [
        int(example[column])
        for column in INTENSITY_COLUMNS
    ]

    return example


train_step1_ds = train_step1_ds.map(
    add_step1_labels
)

val_step1_ds = val_step1_ds.map(
    add_step1_labels
)

test_step1_ds = test_step1_ds.map(
    add_step1_labels
)

train_step2_ds = train_step2_ds.map(
    add_step2_labels
)

val_step2_ds = val_step2_ds.map(
    add_step2_labels
)

test_step2_ds = test_step2_ds.map(
    add_step2_labels
)


# =========================================================
# 13. FORMAT DATASETS
# =========================================================
train_step1_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_step1_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_step1_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

train_step2_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

val_step2_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)

test_step2_ds.set_format(
    type="torch",
    columns=[
        "input_ids",
        "attention_mask",
        "labels"
    ]
)


# =========================================================
# 14. STEP 1 METRICS
# =========================================================
def compute_metrics_step1(eval_pred):
    logits, labels = eval_pred

    probabilities = 1 / (
        1 + np.exp(-logits)
    )

    predictions = (
        probabilities >= 0.5
    ).astype(int)

    f1_macro = f1_score(
        labels,
        predictions,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        labels,
        predictions,
        average="micro",
        zero_division=0
    )

    pearsons = []

    for class_index in range(
        labels.shape[1]
    ):
        true_column = labels[
            :,
            class_index
        ]

        probability_column = probabilities[
            :,
            class_index
        ]

        if (
            np.std(true_column) == 0
            or np.std(probability_column) == 0
        ):
            pearsons.append(0.0)
        else:
            correlation, _ = pearsonr(
                true_column,
                probability_column
            )

            pearsons.append(
                0.0
                if np.isnan(correlation)
                else float(correlation)
            )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": float(
            np.mean(pearsons)
        )
    }


# =========================================================
# 15. STEP 2 MODEL CLASS
# =========================================================
class RobertaStep2IntensityModel(nn.Module):
    def __init__(self):
        super().__init__()

        self.roberta = RobertaModel.from_pretrained(
            "roberta-base"
        )

        hidden_size = (
            self.roberta.config.hidden_size
        )

        self.dropout = nn.Dropout(0.1)

        self.classifier = nn.Linear(
            hidden_size,
            len(EMOTIONS) * 4
        )

    def forward(
        self,
        input_ids=None,
        attention_mask=None,
        labels=None
    ):
        outputs = self.roberta(
            input_ids=input_ids,
            attention_mask=attention_mask
        )

        cls_output = outputs.last_hidden_state[
            :,
            0,
            :
        ]

        cls_output = self.dropout(
            cls_output
        )

        logits = self.classifier(
            cls_output
        )

        logits = logits.view(
            -1,
            len(EMOTIONS),
            4
        )

        loss = None

        if labels is not None:
            loss_function = nn.CrossEntropyLoss()

            loss = loss_function(
                logits.view(-1, 4),
                labels.long().view(-1)
            )

        return {
            "loss": loss,
            "logits": logits
        }


# =========================================================
# 16. STEP 2 METRICS
# =========================================================
def compute_metrics_step2(eval_pred):
    logits, labels = eval_pred

    predictions = np.argmax(
        logits,
        axis=-1
    )

    true_flat = labels.reshape(-1)
    predicted_flat = predictions.reshape(-1)

    f1_macro = f1_score(
        true_flat,
        predicted_flat,
        average="macro",
        zero_division=0
    )

    f1_micro = f1_score(
        true_flat,
        predicted_flat,
        average="micro",
        zero_division=0
    )

    if (
        np.std(true_flat) == 0
        or np.std(predicted_flat) == 0
    ):
        pearson_mean = 0.0
    else:
        correlation, _ = pearsonr(
            true_flat,
            predicted_flat
        )

        pearson_mean = (
            0.0
            if np.isnan(correlation)
            else float(correlation)
        )

    return {
        "f1_macro": f1_macro,
        "f1_micro": f1_micro,
        "pearson_mean": pearson_mean
    }


# =========================================================
# 17. SENTENCE LOGGING HELPERS
# =========================================================
def clean_number(value):
    value = round(
        float(value),
        4
    )

    if value == 0:
        return 0

    if value == 1:
        return 1

    return value


def labels_to_text(
    binary_labels,
    label_names
):
    selected_labels = [
        label_names[index]
        for index, value in enumerate(
            binary_labels
        )
        if int(value) == 1
    ]

    if not selected_labels:
        return "No Emotion"

    return ", ".join(
        selected_labels
    )


# =========================================================
# 18. STEP 1 CALLBACK
# =========================================================
class SaveMetricsCallbackStep1(
    TrainerCallback
):
    def __init__(
        self,
        file_path,
        sentence_log_dir,
        test_dataset,
        test_texts
    ):
        self.file_path = file_path
        self.sentence_log_dir = sentence_log_dir
        self.test_dataset = test_dataset
        self.test_texts = test_texts

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False
        self.processed_epochs = set()

        self.epoch_list = []
        self.train_loss_list = []

        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []

        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):
        if (
            logs is not None
            and "loss" in logs
            and "eval_loss" not in logs
        ):
            self.current_train_loss = float(
                logs["loss"]
            )

    def save_sentence_log(
        self,
        epoch,
        true_labels,
        predicted_labels,
        probabilities
    ):
        output_path = os.path.join(
            self.sentence_log_dir,
            (
                f"RoBERTa_Step1_Epoch_{epoch}"
                "_Sentences.csv"
            )
        )

        rows = []

        for sentence_index, sentence in enumerate(
            self.test_texts
        ):
            row = {
                "sentence_id": sentence_index,
                "sentence": sentence,
                "true_emotions": labels_to_text(
                    true_labels[sentence_index],
                    EMOTIONS
                ),
                "predicted_emotions": labels_to_text(
                    predicted_labels[
                        sentence_index
                    ],
                    EMOTIONS
                )
            }

            for emotion_index, emotion in enumerate(
                EMOTIONS
            ):
                row[f"true_{emotion}"] = int(
                    true_labels[
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"predicted_{emotion}"
                ] = int(
                    predicted_labels[
                        sentence_index,
                        emotion_index
                    ]
                )

                row[f"prob_{emotion}"] = clean_number(
                    probabilities[
                        sentence_index,
                        emotion_index
                    ]
                )

            rows.append(row)

        sentence_df = pd.DataFrame(
            rows
        )

        sentence_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if (
            self._inside_eval
            or metrics is None
            or "eval_loss" not in metrics
        ):
            return

        epoch = int(
            round(
                float(
                    metrics.get(
                        "epoch",
                        state.epoch
                    )
                )
            )
        )

        if epoch in self.processed_epochs:
            return

        self._inside_eval = True

        try:
            train_loss = (
                self.current_train_loss
                if self.current_train_loss is not None
                else ""
            )

            val_loss = float(
                metrics.get(
                    "eval_loss",
                    0.0
                )
            )

            val_f1_macro = float(
                metrics.get(
                    "eval_f1_macro",
                    0.0
                )
            )

            val_f1_micro = float(
                metrics.get(
                    "eval_f1_micro",
                    0.0
                )
            )

            val_pearson_mean = float(
                metrics.get(
                    "eval_pearson_mean",
                    0.0
                )
            )

            prediction_output = (
                self.trainer_ref.predict(
                    self.test_dataset,
                    metric_key_prefix="test"
                )
            )

            test_metrics = (
                prediction_output.metrics
            )

            test_loss = float(
                test_metrics.get(
                    "test_loss",
                    0.0
                )
            )

            test_f1_macro = float(
                test_metrics.get(
                    "test_f1_macro",
                    0.0
                )
            )

            test_f1_micro = float(
                test_metrics.get(
                    "test_f1_micro",
                    0.0
                )
            )

            test_pearson_mean = float(
                test_metrics.get(
                    "test_pearson_mean",
                    0.0
                )
            )

            logits = (
                prediction_output.predictions
            )

            true_labels = (
                prediction_output
                .label_ids
                .astype(int)
            )

            probabilities = 1 / (
                1 + np.exp(-logits)
            )

            predicted_labels = (
                probabilities >= 0.5
            ).astype(int)

            self.epoch_list.append(epoch)
            self.train_loss_list.append(
                train_loss
            )

            self.val_loss_list.append(
                val_loss
            )

            self.val_f1_macro_list.append(
                val_f1_macro
            )

            self.val_f1_micro_list.append(
                val_f1_micro
            )

            self.val_pearson_mean_list.append(
                val_pearson_mean
            )

            self.test_loss_list.append(
                test_loss
            )

            self.test_f1_macro_list.append(
                test_f1_macro
            )

            self.test_f1_micro_list.append(
                test_f1_micro
            )

            self.test_pearson_mean_list.append(
                test_pearson_mean
            )

            report_dict = classification_report(
                true_labels,
                predicted_labels,
                target_names=EMOTIONS,
                zero_division=0,
                output_dict=True
            )

            with open(
                self.file_path,
                "a",
                newline="",
                encoding="utf-8"
            ) as file:
                writer = csv.writer(file)

                writer.writerow([])
                writer.writerow([
                    f"EPOCH {epoch}"
                ])

                writer.writerow([
                    "epoch",
                    "train_loss",
                    "val_loss",
                    "test_loss",
                    "val_f1_macro",
                    "val_f1_micro",
                    "test_f1_macro",
                    "test_f1_micro",
                    "val_pearson_mean",
                    "test_pearson_mean"
                ])

                for index in range(
                    len(self.epoch_list)
                ):
                    writer.writerow([
                        self.epoch_list[index],
                        self.train_loss_list[index],
                        self.val_loss_list[index],
                        self.test_loss_list[index],
                        self.val_f1_macro_list[index],
                        self.val_f1_micro_list[index],
                        self.test_f1_macro_list[index],
                        self.test_f1_micro_list[index],
                        self.val_pearson_mean_list[index],
                        self.test_pearson_mean_list[index]
                    ])

                writer.writerow([])
                writer.writerow([
                    (
                        "FINAL TEST SCORES "
                        f"AFTER EPOCH {epoch}"
                    )
                ])

                writer.writerow([
                    "metric",
                    "value"
                ])

                writer.writerow([
                    "test_loss",
                    test_loss
                ])

                writer.writerow([
                    "test_f1_macro",
                    test_f1_macro
                ])

                writer.writerow([
                    "test_f1_micro",
                    test_f1_micro
                ])

                writer.writerow([
                    "test_pearson_mean",
                    test_pearson_mean
                ])

                writer.writerow([])
                writer.writerow([
                    (
                        "CLASSWISE RESULTS "
                        f"AFTER EPOCH {epoch}"
                    )
                ])

                writer.writerow([
                    "class",
                    "precision",
                    "recall",
                    "f1_score",
                    "correct_predictions",
                    "support"
                ])

                for (
                    class_index,
                    class_name
                ) in enumerate(EMOTIONS):
                    class_result = (
                        report_dict.get(
                            class_name,
                            {}
                        )
                    )

                    correct_predictions = int(
                        np.sum(
                            (
                                true_labels[
                                    :,
                                    class_index
                                ] == 1
                            )
                            &
                            (
                                predicted_labels[
                                    :,
                                    class_index
                                ] == 1
                            )
                        )
                    )

                    writer.writerow([
                        class_name,
                        class_result.get(
                            "precision",
                            ""
                        ),
                        class_result.get(
                            "recall",
                            ""
                        ),
                        class_result.get(
                            "f1-score",
                            ""
                        ),
                        correct_predictions,
                        class_result.get(
                            "support",
                            ""
                        )
                    ])

            sentence_path = (
                self.save_sentence_log(
                    epoch=epoch,
                    true_labels=true_labels,
                    predicted_labels=predicted_labels,
                    probabilities=probabilities
                )
            )

            self.processed_epochs.add(
                epoch
            )

            print(
                f"\nStep 1 epoch {epoch} saved."
            )

            print(
                "Results file:",
                self.file_path
            )

            print(
                "Sentence file:",
                sentence_path
            )

        finally:
            self._inside_eval = False


# =========================================================
# 19. STEP 2 CALLBACK
# =========================================================
class SaveMetricsCallbackStep2(
    TrainerCallback
):
    def __init__(
        self,
        file_path,
        sentence_log_dir,
        step1_trainer,
        test_step1_dataset,
        test_step2_dataset,
        test_texts
    ):
        self.file_path = file_path
        self.sentence_log_dir = sentence_log_dir
        self.step1_trainer = step1_trainer
        self.test_step1_dataset = (
            test_step1_dataset
        )
        self.test_step2_dataset = (
            test_step2_dataset
        )
        self.test_texts = test_texts

        self.trainer_ref = None
        self.current_train_loss = None
        self._inside_eval = False
        self.processed_epochs = set()

        self.epoch_list = []
        self.train_loss_list = []

        self.val_loss_list = []
        self.val_f1_macro_list = []
        self.val_f1_micro_list = []
        self.val_pearson_mean_list = []

        self.test_loss_list = []
        self.test_f1_macro_list = []
        self.test_f1_micro_list = []
        self.test_pearson_mean_list = []

    def on_log(
        self,
        args,
        state,
        control,
        logs=None,
        **kwargs
    ):
        if (
            logs is not None
            and "loss" in logs
            and "eval_loss" not in logs
        ):
            self.current_train_loss = float(
                logs["loss"]
            )

    def make_final_hierarchical_predictions(
        self
    ):
        step1_output = (
            self.step1_trainer.predict(
                self.test_step1_dataset,
                metric_key_prefix="step1_test"
            )
        )

        true_emotions = (
            step1_output
            .label_ids
            .astype(int)
        )

        step1_probabilities = 1 / (
            1
            + np.exp(
                -step1_output.predictions
            )
        )

        predicted_emotions = (
            step1_probabilities >= 0.5
        ).astype(int)

        step2_output = (
            self.trainer_ref.predict(
                self.test_step2_dataset,
                metric_key_prefix="test"
            )
        )

        step2_logits = (
            step2_output.predictions
        )

        true_intensities = (
            step2_output
            .label_ids
            .astype(int)
        )

        test_loss = float(
            step2_output.metrics.get(
                "test_loss",
                0.0
            )
        )

        step2_probabilities = torch.softmax(
            torch.tensor(
                step2_logits
            ),
            dim=-1
        ).numpy()

        step2_predicted_intensities = (
            np.argmax(
                step2_probabilities,
                axis=-1
            )
        )

        final_predicted_intensities = (
            step2_predicted_intensities
            * predicted_emotions
        )

        true_binary_15 = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=int
        )

        predicted_binary_15 = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=int
        )

        combined_probabilities = np.zeros(
            (
                true_intensities.shape[0],
                len(LABELS)
            ),
            dtype=float
        )

        for emotion_index, emotion in enumerate(
            EMOTIONS
        ):
            for level in LEVELS:
                class_index = (
                    emotion_index * 3
                    + level - 1
                )

                true_binary_15[
                    :,
                    class_index
                ] = (
                    true_intensities[
                        :,
                        emotion_index
                    ] == level
                ).astype(int)

                predicted_binary_15[
                    :,
                    class_index
                ] = (
                    final_predicted_intensities[
                        :,
                        emotion_index
                    ] == level
                ).astype(int)

                combined_probabilities[
                    :,
                    class_index
                ] = (
                    step1_probabilities[
                        :,
                        emotion_index
                    ]
                    *
                    step2_probabilities[
                        :,
                        emotion_index,
                        level
                    ]
                )

        return {
            "true_emotions": true_emotions,
            "predicted_emotions": (
                predicted_emotions
            ),
            "step1_probabilities": (
                step1_probabilities
            ),
            "true_intensities": (
                true_intensities
            ),
            "step2_probabilities": (
                step2_probabilities
            ),
            "step2_predicted_intensities": (
                step2_predicted_intensities
            ),
            "final_predicted_intensities": (
                final_predicted_intensities
            ),
            "true_binary_15": (
                true_binary_15
            ),
            "predicted_binary_15": (
                predicted_binary_15
            ),
            "combined_probabilities": (
                combined_probabilities
            ),
            "test_loss": test_loss
        }

    def save_combined_sentence_log(
        self,
        epoch,
        prediction_data
    ):
        output_path = os.path.join(
            self.sentence_log_dir,
            (
                f"RoBERTa_Step2_Epoch_{epoch}"
                "_Combined_Sentences.csv"
            )
        )

        rows = []

        for sentence_index, sentence in enumerate(
            self.test_texts
        ):
            row = {
                "sentence_id": sentence_index,
                "sentence": sentence,
                "true_emotions": labels_to_text(
                    prediction_data[
                        "true_emotions"
                    ][sentence_index],
                    EMOTIONS
                ),
                "predicted_emotions": labels_to_text(
                    prediction_data[
                        "predicted_emotions"
                    ][sentence_index],
                    EMOTIONS
                ),
                "true_combined_labels": labels_to_text(
                    prediction_data[
                        "true_binary_15"
                    ][sentence_index],
                    LABELS
                ),
                "predicted_combined_labels": labels_to_text(
                    prediction_data[
                        "predicted_binary_15"
                    ][sentence_index],
                    LABELS
                )
            }

            for emotion_index, emotion in enumerate(
                EMOTIONS
            ):
                row[f"true_{emotion}"] = int(
                    prediction_data[
                        "true_emotions"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"predicted_{emotion}"
                ] = int(
                    prediction_data[
                        "predicted_emotions"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"step1_prob_{emotion}"
                ] = clean_number(
                    prediction_data[
                        "step1_probabilities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"true_{emotion}_intensity"
                ] = int(
                    prediction_data[
                        "true_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"step2_predicted_{emotion}_intensity"
                ] = int(
                    prediction_data[
                        "step2_predicted_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                row[
                    f"final_predicted_{emotion}_intensity"
                ] = int(
                    prediction_data[
                        "final_predicted_intensities"
                    ][
                        sentence_index,
                        emotion_index
                    ]
                )

                for level in range(4):
                    row[
                        (
                            f"step2_prob_{emotion}"
                            f"_intensity_{level}"
                        )
                    ] = clean_number(
                        prediction_data[
                            "step2_probabilities"
                        ][
                            sentence_index,
                            emotion_index,
                            level
                        ]
                    )

            for (
                class_index,
                class_name
            ) in enumerate(LABELS):
                row[
                    f"combined_prob_{class_name}"
                ] = clean_number(
                    prediction_data[
                        "combined_probabilities"
                    ][
                        sentence_index,
                        class_index
                    ]
                )

            rows.append(row)

        combined_df = pd.DataFrame(
            rows
        )

        combined_df.to_csv(
            output_path,
            index=False,
            encoding="utf-8-sig"
        )

        return output_path

    def on_evaluate(
        self,
        args,
        state,
        control,
        metrics=None,
        **kwargs
    ):
        if (
            self._inside_eval
            or metrics is None
            or "eval_loss" not in metrics
        ):
            return

        epoch = int(
            round(
                float(
                    metrics.get(
                        "epoch",
                        state.epoch
                    )
                )
            )
        )

        if epoch in self.processed_epochs:
            return

        self._inside_eval = True

        try:
            train_loss = (
                self.current_train_loss
                if self.current_train_loss is not None
                else ""
            )

            val_loss = float(
                metrics.get(
                    "eval_loss",
                    0.0
                )
            )

            val_f1_macro = float(
                metrics.get(
                    "eval_f1_macro",
                    0.0
                )
            )

            val_f1_micro = float(
                metrics.get(
                    "eval_f1_micro",
                    0.0
                )
            )

            val_pearson_mean = float(
                metrics.get(
                    "eval_pearson_mean",
                    0.0
                )
            )

            prediction_data = (
                self.make_final_hierarchical_predictions()
            )

            true_binary_15 = (
                prediction_data[
                    "true_binary_15"
                ]
            )

            predicted_binary_15 = (
                prediction_data[
                    "predicted_binary_15"
                ]
            )

            combined_probabilities = (
                prediction_data[
                    "combined_probabilities"
                ]
            )

            test_loss = (
                prediction_data[
                    "test_loss"
                ]
            )

            test_f1_macro = f1_score(
                true_binary_15,
                predicted_binary_15,
                average="macro",
                zero_division=0
            )

            test_f1_micro = f1_score(
                true_binary_15,
                predicted_binary_15,
                average="micro",
                zero_division=0
            )

            pearsons = []

            for class_index in range(
                len(LABELS)
            ):
                true_column = (
                    true_binary_15[
                        :,
                        class_index
                    ]
                )

                probability_column = (
                    combined_probabilities[
                        :,
                        class_index
                    ]
                )

                if (
                    np.std(true_column) == 0
                    or np.std(
                        probability_column
                    ) == 0
                ):
                    pearsons.append(0.0)
                else:
                    correlation, _ = pearsonr(
                        true_column,
                        probability_column
                    )

                    pearsons.append(
                        0.0
                        if np.isnan(correlation)
                        else float(correlation)
                    )

            test_pearson_mean = float(
                np.mean(pearsons)
            )

            self.epoch_list.append(
                epoch
            )

            self.train_loss_list.append(
                train_loss
            )

            self.val_loss_list.append(
                val_loss
            )

            self.val_f1_macro_list.append(
                val_f1_macro
            )

            self.val_f1_micro_list.append(
                val_f1_micro
            )

            self.val_pearson_mean_list.append(
                val_pearson_mean
            )

            self.test_loss_list.append(
                test_loss
            )

            self.test_f1_macro_list.append(
                test_f1_macro
            )

            self.test_f1_micro_list.append(
                test_f1_micro
            )

            self.test_pearson_mean_list.append(
                test_pearson_mean
            )

            report_dict = classification_report(
                true_binary_15,
                predicted_binary_15,
                target_names=LABELS,
                zero_division=0,
                output_dict=True
            )

            with open(
                self.file_path,
                "a",
                newline="",
                encoding="utf-8"
            ) as file:
                writer = csv.writer(file)

                writer.writerow([])
                writer.writerow([
                    f"EPOCH {epoch}"
                ])

                writer.writerow([
                    "epoch",
                    "train_loss",
                    "val_loss",
                    "test_loss",
                    "val_f1_macro",
                    "val_f1_micro",
                    "test_f1_macro",
                    "test_f1_micro",
                    "val_pearson_mean",
                    "test_pearson_mean"
                ])

                for index in range(
                    len(self.epoch_list)
                ):
                    writer.writerow([
                        self.epoch_list[index],
                        self.train_loss_list[index],
                        self.val_loss_list[index],
                        self.test_loss_list[index],
                        self.val_f1_macro_list[index],
                        self.val_f1_micro_list[index],
                        self.test_f1_macro_list[index],
                        self.test_f1_micro_list[index],
                        self.val_pearson_mean_list[index],
                        self.test_pearson_mean_list[index]
                    ])

                writer.writerow([])
                writer.writerow([
                    (
                        "FINAL TEST SCORES "
                        f"AFTER EPOCH {epoch}"
                    )
                ])

                writer.writerow([
                    "metric",
                    "value"
                ])

                writer.writerow([
                    "test_loss",
                    test_loss
                ])

                writer.writerow([
                    "test_f1_macro",
                    test_f1_macro
                ])

                writer.writerow([
                    "test_f1_micro",
                    test_f1_micro
                ])

                writer.writerow([
                    "test_pearson_mean",
                    test_pearson_mean
                ])

                writer.writerow([])
                writer.writerow([
                    (
                        "CLASSWISE RESULTS "
                        f"AFTER EPOCH {epoch}"
                    )
                ])

                writer.writerow([
                    "class",
                    "precision",
                    "recall",
                    "f1_score",
                    "correct_predictions",
                    "support"
                ])

                for (
                    class_index,
                    class_name
                ) in enumerate(LABELS):
                    class_result = (
                        report_dict.get(
                            class_name,
                            {}
                        )
                    )

                    correct_predictions = int(
                        np.sum(
                            (
                                true_binary_15[
                                    :,
                                    class_index
                                ] == 1
                            )
                            &
                            (
                                predicted_binary_15[
                                    :,
                                    class_index
                                ] == 1
                            )
                        )
                    )

                    writer.writerow([
                        class_name,
                        class_result.get(
                            "precision",
                            ""
                        ),
                        class_result.get(
                            "recall",
                            ""
                        ),
                        class_result.get(
                            "f1-score",
                            ""
                        ),
                        correct_predictions,
                        class_result.get(
                            "support",
                            ""
                        )
                    ])

            sentence_path = (
                self.save_combined_sentence_log(
                    epoch=epoch,
                    prediction_data=prediction_data
                )
            )

            self.processed_epochs.add(
                epoch
            )

            print(
                f"\nStep 2 epoch {epoch} saved."
            )

            print(
                "Results file:",
                self.file_path
            )

            print(
                "Combined sentence file:",
                sentence_path
            )

        finally:
            self._inside_eval = False




Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
Using device: cuda

Loading BRIGHTER dataset...
Original sizes:
{'train': 2763, 'val': 115, 'test': 2765}

Emotion labels:
['anger', 'fear', 'joy', 'sadness', 'surprise']

Final 15 labels:
['anger_1', 'anger_2', 'anger_3', 'fear_1', 'fear_2', 'fear_3', 'joy_1', 'joy_2', 'joy_3', 'sadness_1', 'sadness_2', 'sadness_3', 'surprise_1', 'surprise_2', 'surprise_3']

New split sizes:
{'train': 3950, 'val': 1128, 'test': 565}

Sample data:
                                                text  anger  fear  joy  \
0              " She was slapping at air, screaming.      0     1    0   
1  I think it's because I've got one of those lit...      1     1    0   
2  To this day, I still don't know the reasoning ...      1     1    0   
3  I closed my eyes and listened to the silence w...      0     0    1   
4                          Except it was in the sky.      0     1 

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

Map:   0%|          | 0/1128 [00:00<?, ? examples/s]

Map:   0%|          | 0/565 [00:00<?, ? examples/s]

Map:   0%|          | 0/3950 [00:00<?, ? examples/s]

...

Step 1 test sentence-wise log saved at: /content/drive/MyDrive/RoBERTa_Hierarchical_Step1_Test_Sentence_Log.csv


Final hierarchical test sentence-wise log saved at: /content/drive/MyDrive/RoBERTa_Hierarchical_Final_Test_Sentence_Log.csv

Sample Step 1 test sentence-wise log:


,sentence_id,sentence,true_emotions,predicted_emotions,prob_anger,prob_fear,prob_joy,prob_sadness,prob_surprise
0,0,He changed our last name ever-so-slightly and ...,surprise,"anger, surprise",0.5336,0.4288,0.1390,0.4067,0.8043
1,1,They just never went away.,"fear, sadness","fear, sadness",0.2977,0.7504,0.0415,0.8995,0.4050
2,2,"""Well I have loads, you can share mine"".",joy,joy,0.0564,0.0687,0.8171,0.0491,0.1542
3,3,Pretty much everyone objected to my wedding.,"anger, fear, sadness, surprise","anger, fear",0.6968,0.5301,0.0685,0.3455,0.2744
4,4,"I was in better bike shape last year, and had ...",No Emotion,joy,0.0473,0.0804,0.8862,0.0744,0.0865
5,5,"Somehow, while scrubbing the left side of my f...","fear, surprise","fear, surprise",0.1455,0.9578,0.0382,0.3237,0.7193
6,6,At about 4 i was running with a pole with one ...,"fear, surprise","fear, surprise",0.1765,0.9432,0.0589,0.1981,0.8520
7,7,No one else in the class did.,"sadness, surprise","fear, surprise",0.0437,0.5636,0.0703,0.0845,0.5165
8,8,My tongue had sucked it up and the hole was cl...,surprise,No Emotion,0.0277,0.3359,0.3120,0.0853,0.0480
9,9,"Because this is a karma thread, you know how t...",surprise,"anger, fear, surprise",0.6904,0.6074,0.1178,0.3117,0.5875



Sample final hierarchical test sentence-wise log:


,sentence_id,sentence,true_labels,predicted_labels,prob_anger_1,prob_anger_2,prob_anger_3,prob_fear_1,prob_fear_2,prob_fear_3,prob_joy_1,prob_joy_2,prob_joy_3,prob_sadness_1,prob_sadness_2,prob_sadness_3,prob_surprise_1,prob_surprise_2,prob_surprise_3
0,0,He changed our last name ever-so-slightly and ...,surprise_2,surprise_1,0.0419,0.0129,0.0033,0.0347,0.0023,0.0003,0.0520,0.0099,0.0009,0.0170,0.0013,0.0002,0.7271,0.0401,0.0058
1,1,They just never went away.,"fear_1, sadness_2",sadness_1,0.1108,0.0213,0.0094,0.0703,0.0033,0.0004,0.0003,0.0001,0.0001,0.5553,0.2672,0.0444,0.0837,0.0036,0.0015
2,2,"""Well I have loads, you can share mine"".",joy_1,joy_1,0.0003,0.0003,0.0001,0.0006,0.0000,0.0000,0.6979,0.0857,0.0083,0.0003,0.0001,0.0001,0.0018,0.0002,0.0001
3,3,Pretty much everyone objected to my wedding.,"anger_1, fear_1, sadness_2, surprise_1",anger_1,0.2778,0.2634,0.0598,0.0625,0.0026,0.0006,0.0030,0.0006,0.0002,0.0767,0.0176,0.0039,0.0956,0.0079,0.0025
4,4,"I was in better bike shape last year, and had ...",No Emotion,joy_1,0.0001,0.0002,0.0001,0.0010,0.0001,0.0001,0.6225,0.2185,0.0215,0.0014,0.0002,0.0002,0.0001,0.0000,0.0000
5,5,"Somehow, while scrubbing the left side of my f...","fear_1, surprise_1",fear_2,0.0074,0.0014,0.0004,0.1095,0.7453,0.0999,0.0001,0.0000,0.0000,0.1838,0.0359,0.0134,0.3135,0.0394,0.0100
6,6,At about 4 i was running with a pole with one ...,"fear_2, surprise_1","fear_1, surprise_1",0.0092,0.0009,0.0002,0.5526,0.3642,0.0114,0.0006,0.0001,0.0000,0.0270,0.0010,0.0002,0.5078,0.0652,0.0106
7,7,No one else in the class did.,"sadness_1, surprise_1",No Emotion,0.0003,0.0001,0.0000,0.0226,0.0017,0.0004,0.0057,0.0006,0.0002,0.0061,0.0006,0.0002,0.0608,0.0024,0.0014
8,8,My tongue had sucked it up and the hole was cl...,surprise_1,No Emotion,0.0001,0.0001,0.0000,0.0344,0.0015,0.0005,0.0954,0.0071,0.0008,0.0017,0.0002,0.0001,0.0001,0.0000,0.0000
9,9,"Because this is a karma thread, you know how t...",surprise_1,"anger_2, surprise_1",0.3027,0.3084,0.0634,0.2269,0.0344,0.0041,0.0048,0.0011,0.0003,0.0189,0.0040,0.0006,0.3263,0.0501,0.0082


In [ ]:
# =========================================================
# 20. STEP 1 MODEL
# =========================================================
model_step1 = (
    RobertaForSequenceClassification
    .from_pretrained(
        "roberta-base",
        num_labels=len(EMOTIONS),
        problem_type=(
            "multi_label_classification"
        )
    )
)


# =========================================================
# 21. STEP 1 TRAINING ARGUMENTS
# =========================================================
training_args_step1 = TrainingArguments(
    output_dir="/content/roberta_step1_output",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    per_device_eval_batch_size=16,
    num_train_epochs=2,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="f1_macro",
    greater_is_better=True,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 22. STEP 1 TRAINER
# =========================================================
callback_step1 = SaveMetricsCallbackStep1(
    file_path=LOG_FILE_STEP1,
    sentence_log_dir=STEP1_SENTENCE_DIR,
    test_dataset=test_step1_ds,
    test_texts=test_texts
)

trainer_step1 = Trainer(
    model=model_step1,
    args=training_args_step1,
    train_dataset=train_step1_ds,
    eval_dataset=val_step1_ds,
    data_collator=data_collator,
    processing_class=tokenizer,
    compute_metrics=compute_metrics_step1,
    callbacks=[
        callback_step1,
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        )
    ]
)

callback_step1.trainer_ref = (
    trainer_step1
)


# =========================================================
# 23. TRAIN STEP 1
# =========================================================
print(
    "\nStarting Step 1: "
    "Emotion Detection"
)

step1_start_time = time.time()

trainer_step1.train()

step1_end_time = time.time()

print(
    f"Step 1 training time: "
    f"{step1_end_time - step1_start_time:.1f} "
    "seconds"
)

print(
    "Step 1 result file:",
    LOG_FILE_STEP1
)

print(
    "Step 1 sentence logs:",
    STEP1_SENTENCE_DIR
)


# =========================================================
# 24. STEP 2 MODEL INSTANCE
# =========================================================
model_step2 = (
    RobertaStep2IntensityModel()
)


# =========================================================
# 25. STEP 2 TRAINING ARGUMENTS
# =========================================================
training_args_step2 = TrainingArguments(
    output_dir="/content/roberta_step2_output",
    learning_rate=2e-5,
    per_device_train_batch_size=8,
    per_device_eval_batch_size=8,
    num_train_epochs=10,
    eval_strategy="epoch",
    logging_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    save_total_limit=2,
    report_to="none",
    fp16=torch.cuda.is_available(),
    seed=SEED
)


# =========================================================
# 26. STEP 2 TRAINER
# =========================================================
callback_step2 = SaveMetricsCallbackStep2(
    file_path=LOG_FILE_STEP2,
    sentence_log_dir=STEP2_SENTENCE_DIR,
    step1_trainer=trainer_step1,
    test_step1_dataset=test_step1_ds,
    test_step2_dataset=test_step2_ds,
    test_texts=test_texts
)

trainer_step2 = Trainer(
    model=model_step2,
    args=training_args_step2,
    train_dataset=train_step2_ds,
    eval_dataset=val_step2_ds,
    data_collator=data_collator,
    compute_metrics=compute_metrics_step2,
    callbacks=[
        callback_step2,
        EarlyStoppingCallback(
            early_stopping_patience=1,
            early_stopping_threshold=0.0
        )
    ]
)

callback_step2.trainer_ref = (
    trainer_step2
)


# =========================================================
# 27. TRAIN STEP 2
# =========================================================
print(
    "\nStarting Step 2: "
    "Intensity Classification"
)

step2_start_time = time.time()

trainer_step2.train()

step2_end_time = time.time()

print(
    f"Step 2 training time: "
    f"{step2_end_time - step2_start_time:.1f} "
    "seconds"
)

print(
    "Step 2 result file:",
    LOG_FILE_STEP2
)

print(
    "Step 2 combined sentence logs:",
    STEP2_SENTENCE_DIR
)


# =========================================================
# 28. DISPLAY CREATED FILES
# =========================================================
print("\nStep 1 sentence files:")

for file_name in sorted(
    os.listdir(STEP1_SENTENCE_DIR)
):
    if file_name.endswith(".csv"):
        print(
            os.path.join(
                STEP1_SENTENCE_DIR,
                file_name
            )
        )

print("\nStep 2 combined sentence files:")

for file_name in sorted(
    os.listdir(STEP2_SENTENCE_DIR)
):
    if file_name.endswith(".csv"):
        print(
            os.path.join(
                STEP2_SENTENCE_DIR,
                file_name
            )
        )